In [ ]:
# Script para copiar todos os arquivos .gff gerados pelo ISEScan para um único diretório
nano copy_gff_files.sh
## Copiar o script abaixo

## Script #########################################################################################################################################

#!/bin/bash

# Diretório de origem (onde estão as pastas com os genomas)
SOURCE_DIR="/mnt/dados/victor_baca/outputs/isescan"

# Diretório de destino (onde os arquivos .gff serão copiados)
DEST_DIR="/mnt/dados/victor_baca/outputs/ISEScan_gff"

# Criar o diretório de destino se não existir
mkdir -p "$DEST_DIR"

# Encontrar e copiar todos os arquivos .gff
find "$SOURCE_DIR" -type f -name "*.gff" -exec cp {} "$DEST_DIR" \;

echo "Cópia concluída!"
echo "Arquivos .gff copiados para: $DEST_DIR"

##################################################################################################################################################

chmod +x copy_gff_files.sh # Dar permissão de execução 
./copy_gff_files.sh # Executa o script

In [ ]:
# Criar script para deixar os nomes dos arquivos dos diretorios dos gffs iguais

nano rename_gff_files.py
## Copiar o script abaixo

## Script #########################################################################################################################################

import os
import glob

# Caminho do diretório com os arquivos
diretorio = "/mnt/dados/victor_baca/outputs/ISEScan_gff"

# Mudar para o diretório
os.chdir(diretorio)

# Buscar todos os arquivos que contêm "_blast" no nome
arquivos = glob.glob("*.fna.gff")

# Contador para acompanhar o progresso
contador = 0

print(f"Total de arquivos encontrados: {len(arquivos)}\n")

# Renomear cada arquivo
for arquivo_antigo in arquivos:
    # Criar o novo nome removendo o sufixo indesejado
    arquivo_novo = arquivo_antigo.replace(".fna", "") 
    
    try:
        # Renomear o arquivo
        os.rename(arquivo_antigo, arquivo_novo)
        contador += 1
        print(f"✓ Renomeado: {arquivo_antigo} → {arquivo_novo}")
    except Exception as e:
        print(f"✗ Erro ao renomear {arquivo_antigo}: {e}")

print(f"\n{contador} arquivos renomeados com sucesso!")

##################################################################################################################################################

chmod +x rename_gff_files.py # Dar permissão de execução
python3 rename_gff_files.py # Executa o script

# Para modificar o script e so mudar as variaveis: 
## diretorio 
## arquivos
## arquivo_novo

In [ ]:
# Agora vamos unir os arquivos gff pelo nome em comum dos arquivos 

mkdir BLAST+ISEScan # Criar diretório para salvar os arquivos unidos 

# Agora vamos criar um Script para unir arquivos GFF de dois diretórios pelo nome em comum e salvar no diretório de saída.
nano merge_gff.py

## Script #########################################################################################################################################

#!/usr/bin/env python3

import os
from pathlib import Path

# Definir diretórios
blast_dir = Path("/mnt/dados/victor_baca/outputs/BLAST_gff")
isescan_dir = Path("/mnt/dados/victor_baca/outputs/ISEScan_gff")
output_dir = Path("/mnt/dados/victor_baca/outputs/BLAST+ISEScan")

# Criar diretório de saída se não existir
output_dir.mkdir(parents=True, exist_ok=True)

# Listar arquivos de cada diretório
blast_files = {f.name: f for f in blast_dir.glob("*.gff")}
isescan_files = {f.name: f for f in isescan_dir.glob("*.gff")}

# Encontrar arquivos em comum
common_files = set(blast_files.keys()) & set(isescan_files.keys())

print(f"Arquivos encontrados no BLAST_gff: {len(blast_files)}")
print(f"Arquivos encontrados no ISEScan_gff: {len(isescan_files)}")
print(f"Arquivos em comum: {len(common_files)}")
print("\nProcessando arquivos...\n")

# Processar cada arquivo em comum
for filename in sorted(common_files):
    blast_file = blast_files[filename]
    isescan_file = isescan_files[filename]
    output_file = output_dir / filename
    
    print(f"Unindo: {filename}")
    
    try:
        # Ler conteúdo dos dois arquivos
        with open(blast_file, 'r') as f:
            blast_content = f.read()
        
        with open(isescan_file, 'r') as f:
            isescan_content = f.read()
        
        # Escrever arquivo combinado
        with open(output_file, 'w') as f:
            f.write(f"# Arquivo combinado de BLAST e ISEScan\n")
            f.write(f"# BLAST source: {blast_file}\n")
            f.write(f"# ISEScan source: {isescan_file}\n")
            f.write("#" + "="*70 + "\n")
            f.write("# BLAST annotations\n")
            f.write("#" + "="*70 + "\n")
            f.write(blast_content)
            if not blast_content.endswith('\n'):
                f.write('\n')
            f.write("#" + "="*70 + "\n")
            f.write("# ISEScan annotations\n")
            f.write("#" + "="*70 + "\n")
            f.write(isescan_content)
        
        print(f"  ✓ Salvo em: {output_file}")
        
    except Exception as e:
        print(f"  ✗ Erro ao processar {filename}: {e}")

print(f"\n{'='*70}")
print(f"Processo concluído!")
print(f"Total de arquivos unidos: {len(common_files)}")
print(f"Diretório de saída: {output_dir}")

##################################################################################################################################################

chmod +x merge_gff.py # Dar permissão de execução
python3 merge_gff.py # Executa o script

In [ ]:
# Agora vamos usar o bedtools para processar os arquivos unidos 

## 1 - Criar diretório para salvar os arquivos processados pelo bedtools
mkdir bedtools

## 2 - Instalar o bedtools

### Instalação via apt-get (no Ubuntu/Debian) 
sudo apt-get update
sudo apt-get install bedtools

### Instalação via conda (recomendado) 
conda create -n bedtools
conda activate bedtools # Ativar o ambiente bedtools 
conda install -c bioconda -c conda-forge bedtools # Instalar o bedtools

## 3 - Verificar a instalação
bedtools --version

# O bedtools vai servir para retirar as redundâncias dos arquivos gff unidos

In [ ]:
# Maneira manual de usar o bedtools para processar os arquivos gff unidos

sort -k1,1 -k4,4n file.gff > file.gff
bedtools merge -i file.gff -s -c 7,1 -o distinct,count > file.bed
awk -F"\t" '{print $1 "\t" "." "\t" "." "\t" $2+1 "\t" $3 "\t" "." "\t" $4 "\t" "." "\t" "." "\t" $5}' file.bed > file_dois.gff
bedtools intersect -a file_dois.gff -b file.gff -wa -wb -s > file_final.gff

In [ ]:
# Maneira automatizada de usar o bedtools para processar todos os arquivos gff unidos

nano process_all_gff.sh # Criar o script
# Copiar o script abaixo no arquivo

## Script #########################################################################################################################################

#!/bin/bash

# Diretórios
INPUT_DIR="/mnt/dados/victor_baca/outputs/BLAST+ISEScan"
OUTPUT_DIR="/mnt/dados/victor_baca/outputs/bedtools"

# Criar diretório de saída se não existir
mkdir -p "$OUTPUT_DIR"

# Processar cada arquivo .gff no diretório de entrada
for file in "$INPUT_DIR"/*.gff; do
    # Extrair o nome base do arquivo (sem o caminho)
    basename=$(basename "$file")
    filename="${basename%.gff}"
    
    echo "Processando: $basename"
    
    # Definir nomes dos arquivos temporários e finais
    sorted_gff="$OUTPUT_DIR/${filename}_sorted.gff"
    merged_bed="$OUTPUT_DIR/${filename}_merged.bed"
    dois_gff="$OUTPUT_DIR/${filename}_dois.gff"
    final_gff="$OUTPUT_DIR/${filename}_final.gff"
    
    # Executar os comandos do bedtools
    # 1. Ordenar o arquivo GFF
    sort -k1,1 -k4,4n "$file" > "$sorted_gff"
    
    # 2. Fazer merge com bedtools
    bedtools merge -i "$sorted_gff" -s -c 7,1 -o distinct,count > "$merged_bed"
    
    # 3. Processar com awk
    awk -F"\t" '{print $1 "\t" "." "\t" "." "\t" $2+1 "\t" $3 "\t" "." "\t" $4 "\t" "." "\t" "." "\t" $5}' "$merged_bed" > "$dois_gff"
    
    # 4. Fazer intersect final
    bedtools intersect -a "$dois_gff" -b "$sorted_gff" -wa -wb -s > "$final_gff"
    
    # Remover arquivos intermediários (opcional - comente se quiser mantê-los)
    rm "$sorted_gff" "$merged_bed" "$dois_gff"
    
    echo "Concluído: $final_gff"
    echo "---"
done

echo "Processamento completo! Arquivos salvos em: $OUTPUT_DIR"

##################################################################################################################################################

chmod +x process_all_gff.sh # Dar permissão de execução
./process_all_gff.sh # Executa o script

In [ ]:
# Analise rápida do arquivo final
awk '{for(i=9;i<=NF;i++) printf "%s%s", $i, (i<NF ? "\t" : "\n")}' GCF_019794875.1_ASM1979487v1* | column -t -s $'\t'